In [6]:
# google_trends_crawler.py

from pytrends.request import TrendReq
import pandas as pd
import time
import os

# ── 初始化 ──────────────────────────────────────────────────
pytrends = TrendReq(
    hl="zh-TW",     # 語言：繁體中文
    tz=480,         # 時區：台灣 UTC+8
    timeout=(10, 25),
    retries=3,
    backoff_factor=0.5
)

os.makedirs("output/csv", exist_ok=True)

# ── 功能一：各銀行品牌搜尋趨勢 ─────────────────────────────
def get_bank_brand_trend():
    """
    查詢各銀行信用卡關鍵字的近5年搜尋趨勢
    注意：pytrends 每次最多查 5 個關鍵字
    """
    keywords = [
        "國泰世華信用卡",
        "富邦信用卡",
        "中信信用卡",
        "玉山信用卡",
        "台新信用卡"
    ]

    # 建立查詢
    pytrends.build_payload(
        kw_list=keywords,
        cat=0,
        timeframe="today 5-y",   # 近5年
        geo="TW",                # 台灣
        gprop=""
    )

    df = pytrends.interest_over_time()

    # 移除 isPartial 欄位
    if "isPartial" in df.columns:
        df = df.drop(columns=["isPartial"])

    print("✅ 銀行品牌趨勢：")
    print(df.tail())

    df.to_csv("output/csv/bank_brand_trend.csv", encoding="utf-8-sig")
    return df


# ── 功能二：信用卡產品類型趨勢 ──────────────────────────────
def get_product_type_trend():
    """
    查詢不同類型信用卡的搜尋趨勢
    """
    keywords = [
        "現金回饋卡",
        "哩程卡",
        "網購信用卡",
        "信用卡推薦",
        "免年費信用卡"
    ]

    pytrends.build_payload(
        kw_list=keywords,
        timeframe="today 5-y",
        geo="TW"
    )

    df = pytrends.interest_over_time()
    if "isPartial" in df.columns:
        df = df.drop(columns=["isPartial"])

    df.to_csv("output/csv/product_type_trend.csv", encoding="utf-8-sig")
    return df


# ── 功能三：各縣市搜尋熱度分布 ──────────────────────────────
def get_region_interest(keyword="信用卡推薦"):
    """
    查詢某關鍵字在台灣各縣市的搜尋熱度
    """
    pytrends.build_payload(
        kw_list=[keyword],
        timeframe="today 12-m",
        geo="TW"
    )

    df = pytrends.interest_by_region(
        resolution="CITY",      # 縣市層級
        inc_low_vol=True,       # 包含低搜尋量地區
        inc_geo_code=False
    )

    # 只取有資料的縣市
    df = df[df[keyword] > 0].sort_values(keyword, ascending=False)

    print(f"\n✅ '{keyword}' 各縣市熱度：")
    print(df)

    df.to_csv(f"output/csv/region_{keyword}.csv", encoding="utf-8-sig")
    return df


# ── 功能四：相關搜尋詞 ───────────────────────────────────────
def get_related_queries(keyword="玉山信用卡"):
    """
    查詢某關鍵字的相關搜尋詞（熱門 + 快速上升）
    """
    pytrends.build_payload(
        kw_list=[keyword],
        timeframe="today 12-m",
        geo="TW"
    )

    related = pytrends.related_queries()

    top_df    = related[keyword]["top"]     # 最熱門相關詞
    rising_df = related[keyword]["rising"]  # 快速上升詞

    print(f"\n✅ '{keyword}' 熱門相關詞：")
    print(top_df)

    print(f"\n✅ '{keyword}' 快速上升詞：")
    print(rising_df)

    if top_df is not None:
        top_df.to_csv(f"output/csv/related_top_{keyword}.csv",
                      encoding="utf-8-sig", index=False)
    if rising_df is not None:
        rising_df.to_csv(f"output/csv/related_rising_{keyword}.csv",
                         encoding="utf-8-sig", index=False)

    return top_df, rising_df


# ── 功能五：爬取多組關鍵字（超過5個時分批查詢）────────────────
def get_multiple_keywords():
    """
    超過5個關鍵字時，分批查詢再合併
    """
    all_keywords = [
        "國泰世華信用卡", "富邦信用卡", "中信信用卡",
        "玉山信用卡",     "台新信用卡", "永豐信用卡",
        "聯邦信用卡",     "星展信用卡"
    ]

    # 每批最多 5 個
    batch_size = 5
    all_dfs = []

    for i in range(0, len(all_keywords), batch_size):
        batch = all_keywords[i:i + batch_size]
        print(f"查詢批次 {i//batch_size + 1}：{batch}")

        pytrends.build_payload(
            kw_list=batch,
            timeframe="today 5-y",
            geo="TW"
        )

        df = pytrends.interest_over_time()
        if "isPartial" in df.columns:
            df = df.drop(columns=["isPartial"])

        all_dfs.append(df)
        time.sleep(3)   # 批次間隔，避免被限流

    # 合併所有批次
    result = pd.concat(all_dfs, axis=1)
    result.to_csv("output/csv/all_banks_trend.csv", encoding="utf-8-sig")
    return result


# ── 主程式 ───────────────────────────────────────────────────
if __name__ == "__main__":

    print("=== 1. 銀行品牌趨勢 ===")
    brand_df = get_bank_brand_trend()
    time.sleep(3)

    print("\n=== 2. 產品類型趨勢 ===")
    product_df = get_product_type_trend()
    time.sleep(3)

    print("\n=== 3. 各縣市熱度 ===")
    region_df = get_region_interest("信用卡推薦")
    time.sleep(3)

    print("\n=== 4. 相關搜尋詞 ===")
    top, rising = get_related_queries("玉山信用卡")
    time.sleep(3)

    print("\n=== 5. 多銀行合併查詢 ===")
    all_df = get_multiple_keywords()

    print("\n✅ 所有資料已儲存至 output/csv/")

=== 1. 銀行品牌趨勢 ===


TypeError: Retry.__init__() got an unexpected keyword argument 'method_whitelist'